# Unlabelled autoencoder on the T = 10 benign + GridSybil windows (all/T10_benign_grid)

The unlabelled cell of the baseline grid at T = 10: the protocol of `task/autoencoder_t32` with the
depth-configurable ConvAutoencoder of `task/autoencoder_benign_t20` (14 × 10 in, z = 16, 1 stride-2 stage(s):
encoder strides 1, 2, 1; decoder ConvTranspose k3 s1 (length kept), k4 s2; 52,861 parameters; same optimiser / LR schedule / early stopping as T = 32) trained on **every train
window pooled, benign and GridSybil mixed, with no label read**, on
`src/data/encoder_input/all/T10_benign_grid/` (benign of all 8 scenario folders + GridSybil; train / val / test =
85 / 7.5 / 7.5 by vehicle key, fixed on disk, the all/T32_benign_grid split). Val (mixed, unlabelled) drives early
stopping, the LR schedule and the p95 / p99 alarm thresholds. The test split is scored first; its labels are read
afterwards (section 5).

Two label-free scores per window: the reconstruction error (masked MSE over real rows) and the **Mahalanobis
distance** of z to a Gaussian fitted on the train z. Each is also aggregated per **track** (one link): max and mean.

One seed per run folder, a side experiment, not a thesis result (RULE 1). This notebook runs from the repo root and
reads the run folder written by `python -m task.autoencoder_t10.train` and `python -m task.autoencoder_t10.evaluate`.
Sections: 1 input, 2 model, 3 training record, 4 reconstruction on val, 5 test, 6 context, 7 analysis.

Setup: repo-root check, package imports, the run folder (the newest `runs/*/evaluation.json`).

In [ ]:
import json
from pathlib import Path

assert Path("task/autoencoder_t10/config.py").is_file(), "run this notebook from the repo root"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

from task.autoencoder_benign_t20.model import ConvAutoencoder, count_params
from task.autoencoder_t10.config import TOTAL_PARAMS, AutoencoderT10Config
from task.autoencoder_t10.data import UnlabelledWindows
from task.autoencoder_t10.labels import WindowLabels
from task.autoencoder_t10.scoring import SCORES, Evaluation, Mahalanobis, ScoreFiles, TrackScores, quantile_thresholds
from task.autoencoder_t10.trainer import Trainer

pd.set_option("display.width", 200, "display.max_columns", 30)
RUN = max(Path("task/autoencoder_t10/runs").glob("*/evaluation.json"), key=lambda p: p.stat().st_mtime).parent
settings = AutoencoderT10Config.from_dict(json.loads((RUN / "config.json").read_text()))
summary = json.loads((RUN / "summary.json").read_text())
stored = json.loads((RUN / "evaluation.json").read_text())
env = json.loads((RUN / "env.json").read_text())
META = json.loads((Path(settings.data_dir) / "metadata.json").read_text())
FEATURES = META["features"]
print("run folder:", RUN, "| seed", settings.seed, "| device", env["device"], "| torch", env["torch"])
print("labels read in training:", summary["labels_read"], "| label-file guard:", summary["label_file_guard"])

## 1. Input

Train and val as the training code sees them: `UnlabelledWindows` reads the feature shards (id, x, mask) only and has
no label or info attribute. The class counts of the splits are shown in section 5, from the info files.

In [ ]:
train = UnlabelledWindows.load(settings.data_dir, "train", settings.workers, cache_dir=settings.cache_dir)
val = UnlabelledWindows.load(settings.data_dir, "val", settings.workers, cache_dir=settings.cache_dir)
print("attributes of the training dataset object:", UnlabelledWindows.__slots__)
assert not (train.vehicles() & val.vehicles()), "a sender vehicle is in both train and val"
rows = []
for d in (train, val):
    n = d.n_messages
    rows.append(
        {
            "split": d.split,
            "windows": len(d),
            "tracks": len(set(d.tracks().tolist())),
            "vehicles": len(d.vehicles()),
            "median rows": np.median(n),
            "mean rows": round(n.mean(), 2),
            f"share = {settings.seq_len}": round(float((n == settings.seq_len).mean()), 3),
            "share 1-3": round(float((n <= 3).mean()), 3),
            "padding share": round(1 - d.mask.mean(), 3),
        }
    )
display(pd.DataFrame(rows).set_index("split"))
print(
    "window shape",
    train.x.shape[1:],
    "+ mask; model input (channels x time):",
    (settings.n_features + 1, settings.seq_len),
)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(train.n_messages, bins=np.arange(1, settings.seq_len + 2) - 0.5, color="tab:blue")
ax.set_xlabel("real messages per window")
ax.set_ylabel("train windows")
ax.set_yscale("log")
ax.set_title("Train (unlabelled mix): real messages per window")
plt.show()

Per-feature statistics over the **real** train rows (normalised with the train mean / std, so ≈ 0 / 1).

In [ ]:
real = train.x[train.mask]
pd.DataFrame({"mean": real.mean(0), "std": real.std(0), "min": real.min(0), "max": real.max(0)}, index=FEATURES).round(
    3
)

## 2. Model

The depth-configurable ConvAutoencoder of `task/autoencoder_benign_t20` with n_down = 1 (encoder strides 1, 2, 1; decoder ConvTranspose k3 s1 (length kept), k4 s2): layers
with their output shapes for a batch of 2 windows, and the trainable parameter count (asserted). Every other
setting equals the one-class T = 32 package's (checked below).

In [ ]:
model = ConvAutoencoder(settings)
x0, m0 = torch.zeros(2, settings.seq_len, settings.n_features), torch.ones(2, settings.seq_len, dtype=torch.bool)
table, hooks = [], []
layer_types = (torch.nn.Conv1d, torch.nn.ConvTranspose1d, torch.nn.Linear, torch.nn.BatchNorm1d)
for name, module in model.named_modules():
    if isinstance(module, layer_types):
        hooks.append(
            module.register_forward_hook(
                lambda mod, inp, out, name=name: table.append(
                    {
                        "layer": name,
                        "type": type(mod).__name__,
                        "output shape": tuple(out.shape),
                        "parameters": sum(p.numel() for p in mod.parameters()),
                    }
                )
            )
        )
model.eval()
with torch.no_grad():
    model(x0, m0)
for h in hooks:
    h.remove()
display(pd.DataFrame(table).set_index("layer"))
assert count_params(model) == TOTAL_PARAMS == summary["params"]
print(f"trainable parameters: {count_params(model):,}")
same = (
    "lr",
    "weight_decay",
    "batch_size",
    "max_epochs",
    "patience",
    "plateau_factor",
    "plateau_patience",
    "min_delta",
    "grad_clip",
    "latent_dim",
)
from task.autoencoder_benign_t32.config import AutoencoderConfig

print(
    "settings equal to the one-class package:",
    all(getattr(settings, k) == getattr(AutoencoderConfig(), k) for k in same),
)
pd.Series({k: getattr(settings, k) for k in same}, name="value").to_frame().T

## 3. Training record

Loss = masked MSE over real rows; Adam; ReduceLROnPlateau and early stopping on the val loss (relative min_delta).

In [ ]:
history = pd.read_json(RUN / "history.jsonl", lines=True)
print(
    f"best epoch {summary['best_epoch']} (val masked MSE {summary['best_val_loss']:.5f}); epochs run {summary['epochs_run']}; "
    f"stop: {summary['stop_reason']}; LR halvings {len(summary['lr_changes'])}; "
    f"training {summary['train_seconds'] / 60:.1f} min, whole train.py {summary['wall_seconds'] / 60:.1f} min on {summary['device']}; "
    f"median {history.seconds.median():.1f} s / epoch"
)
best_row = history[history.epoch == summary["best_epoch"]].iloc[0]
print(
    f"at the best epoch: train {best_row.train_loss:.5f}, val {best_row.val_loss:.5f} (val / train {best_row.val_loss / best_row.train_loss:.2f})"
)
Trainer.plot(history.to_dict("records"), summary["best_epoch"], summary["lr_changes"], RUN / "curves.png")
display(pd.concat([history.head(3), history.tail(3)]).set_index("epoch").round(5))
from IPython.display import Image

Image(str(RUN / "curves.png"), width=700)

## 4. Reconstruction on val (unlabelled)

The val scores written by `train.py`, re-computed here from `best.pt`; the val p95 / p99 thresholds re-derived and
compared with `summary.json`; three full val windows with their reconstruction; mean error per feature.

In [ ]:
trainer = Trainer.from_checkpoint(RUN / "best.pt")
maha = Mahalanobis.from_state(np.load(RUN / "mahalanobis.npz"))
files = ScoreFiles(RUN)
val_out = trainer.score(val.x, val.mask, keep_recon=True)
val_saved = files.load("val")
assert np.array_equal(val_saved["ids"], val.ids)
assert np.allclose(val_saved["reconstruction"], val_out["error"], rtol=1e-3, atol=1e-5)
assert np.allclose(val_saved["mahalanobis"], maha.score(val_out["z"]), rtol=1e-3, atol=1e-4)
pooled = float((val_out["error"].astype(np.float64) * val.n_messages).sum() / val.n_messages.sum())
print(
    f"val windows {len(val):,}; pooled masked MSE {pooled:.5f} (run's best val loss {summary['best_val_loss']:.5f}); "
    f"mahalanobis: cond {summary['mahalanobis']['cond']:.1f}, shrinkage {summary['mahalanobis']['shrinkage']:g}"
)
val_tracks = TrackScores(val.tracks())
for name in SCORES:
    q = quantile_thresholds(val_saved[name], settings.threshold_quantiles)
    assert all(np.isclose(q[k], summary["thresholds"]["window"][name][k], rtol=1e-5) for k in q)
    for how, agg in val_tracks.aggregate(val_saved[name]).items():
        q = quantile_thresholds(agg, settings.threshold_quantiles)
        assert all(np.isclose(q[k], summary["thresholds"]["track"][f"{name}_{how}"][k], rtol=1e-5) for k in q)
print("val thresholds re-derived from the score files = summary.json")
display(pd.DataFrame(summary["thresholds"]["window"]).T.round(4))

show = ["claimed_pos_x", "claimed_vel_x", "range"]
picks = np.flatnonzero(val.n_messages == settings.seq_len)[:3]
fig, axes = plt.subplots(len(picks), len(show), figsize=(12, 2.4 * len(picks)), squeeze=False)
for r, i in enumerate(picks):
    for c, f in enumerate(show):
        j = FEATURES.index(f)
        axes[r, c].plot(val.x[i, :, j], label="input")
        axes[r, c].plot(val_out["recon"][i, :, j], "--", label="reconstruction")
        axes[r, c].set_title(f, fontsize=9)
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()
pd.Series(val_out["error_feature"].mean(0), index=FEATURES, name="val MSE").sort_values(ascending=False).round(
    4
).to_frame().T

## 5. Test

The test windows are read **without labels**, scored with `best.pt`, and checked against the score files written by
`evaluate.py` (`scores/test_*.npy`, aligned with `test_ids.npy`). No threshold or direction is chosen here: the
thresholds are the val p95 / p99 above, the length rule is score = −n_messages.

In [ ]:
test = UnlabelledWindows.load(settings.data_dir, "test", settings.workers, cache_dir=settings.cache_dir)
test_out = trainer.score(test.x, test.mask)
saved = files.load("test")
assert np.array_equal(saved["ids"], test.ids) and np.array_equal(saved["n_messages"], test.n_messages)
assert np.allclose(saved["reconstruction"], test_out["error"], rtol=1e-3, atol=1e-5)
assert np.allclose(saved["mahalanobis"], maha.score(test_out["z"]), rtol=1e-3, atol=1e-4)
assert not (test.vehicles() & (train.vehicles() | val.vehicles())), "a test vehicle is in train or val"
print(f"test windows {len(test):,}; score files match a fresh scoring pass; no test vehicle in train / val")

Now the labels (`_info.json`, positive = GridSybil). The report is rebuilt with `Evaluation` from the score files and
compared with the stored `evaluation.json`. Class counts per split come from the info files (train / val labels are
read here for counting only).

In [ ]:
labels = WindowLabels.load(settings.data_dir, "test", settings.workers)
labels.check(saved["ids"], saved["n_messages"])
ev = Evaluation(
    {n: saved[n] for n in SCORES},
    labels.y,
    saved["n_messages"],
    test.tracks(),
    labels.group,
    labels.scenario,
    summary["thresholds"],
)
grid_rows = labels.family == "GridSybil"
reports = {"all test windows": ev.report(), "GridSybil folders only": ev.report(grid_rows)}


def close(a, b):
    if isinstance(a, dict):
        return set(a) == set(b) and all(close(a[k], b[k]) for k in a)
    if isinstance(a, float):
        return bool(np.isclose(a, b, rtol=1e-9, atol=1e-12, equal_nan=True))
    return a == b


assert close(reports["all test windows"], stored["test_all"]) and close(
    reports["GridSybil folders only"], stored["test_gridsybil_folders"]
)
print("report rebuilt here = evaluation.json")
counts = pd.DataFrame(stored["class_counts"]).T[["benign", "gridsybil", "gridsybil_share"]]
display(counts.round(4))
pd.DataFrame(
    {
        k: {
            "windows": r["windows"],
            "benign": r["benign"],
            "GridSybil": r["gridsybil"],
            "GridSybil share": round(r["gridsybil_share"], 4),
            "1-3-row share": round(r["share_rows_1_3"], 4),
            "tracks": r["track"]["reconstruction"]["tracks"],
            "mixed-label tracks": r["track"]["reconstruction"]["mixed_label_tracks"],
        }
        for k, r in reports.items()
    }
).T

Window level, GridSybil vs benign: AUROC, PR-AUC (chance = GridSybil share), recall and benign FPR at the val p95 /
p99 thresholds; AUROC on the 1–3-row and the ≥ 4-row windows; per group. **Length alone** uses the fixed rule
score = −n_messages (shorter = more anomalous, the convention of `task/autoencoder_benign_t32`); the last column gives
what the `task/autoencoder_st_t32` convention (max(a, 1 − a), direction picked on the evaluated labels) would report.

In [ ]:
rows = []
for subset, r in reports.items():
    for name in SCORES:
        w = r["window"][name]
        rows.append(
            {
                "subset": subset,
                "score": name,
                "AUROC": w["auroc"],
                "PR-AUC": w["pr_auc"],
                "chance": w["chance"],
                "recall p95": w["p95"]["recall"],
                "benign FPR p95": w["p95"]["benign_fpr"],
                "recall p99": w["p99"]["recall"],
                "benign FPR p99": w["p99"]["benign_fpr"],
                "AUROC 1-3 rows": w["rows_1_3"]["auroc"],
                "AUROC >= 4 rows": w["rows_ge_4"]["auroc"],
                "AUROC 0709": w["by_group"]["0709"]["auroc"],
                "AUROC 1416": w["by_group"]["1416"]["auroc"],
            }
        )
    lo = r["length_only"]
    rows.append(
        {
            "subset": subset,
            "score": "length alone (-n_messages)",
            "AUROC": lo["auroc"],
            "PR-AUC": lo["pr_auc"],
            "chance": r["gridsybil_share"],
            "AUROC best direction (st_t32 convention)": lo["auroc_best_direction"],
        }
    )
window_table = pd.DataFrame(rows).set_index(["subset", "score"])
window_table.round(4)

Sanity check of the headline numbers with plain sklearn calls on the score files.

In [ ]:
for name in SCORES:
    a, p = roc_auc_score(labels.y, saved[name]), average_precision_score(labels.y, saved[name])
    assert np.isclose(a, reports["all test windows"]["window"][name]["auroc"]) and np.isclose(
        p, reports["all test windows"]["window"][name]["pr_auc"]
    )
    print(
        f"{name}: AUROC {a:.4f}, PR-AUC {p:.4f}; GridSybil folders only: AUROC {roc_auc_score(labels.y[grid_rows], saved[name][grid_rows]):.4f}"
    )
print(f"length alone, fixed rule: AUROC {roc_auc_score(labels.y, -saved['n_messages'].astype(float)):.4f}")

Per track (max / mean of a link's window scores), with the val track thresholds; per group counts and metrics.

In [ ]:
rows = []
for subset, r in reports.items():
    for name in SCORES:
        for how in ("max", "mean"):
            t = r["track"][name][how]
            rows.append(
                {
                    "subset": subset,
                    "score": name,
                    "level": f"track {how}",
                    "tracks": r["track"][name]["tracks"],
                    "AUROC": t["auroc"],
                    "PR-AUC": t["pr_auc"],
                    "chance": t["chance"],
                    "recall p95": t["p95"]["recall"],
                    "benign FPR p95": t["p95"]["benign_fpr"],
                    "recall p99": t["p99"]["recall"],
                    "benign FPR p99": t["p99"]["benign_fpr"],
                }
            )
display(pd.DataFrame(rows).set_index(["subset", "score", "level"]).round(4))
rows = []
for subset, r in reports.items():
    for name in SCORES:
        for g, d in r["window"][name]["by_group"].items():
            rows.append(
                {
                    "subset": subset,
                    "score": name,
                    "group": g,
                    "benign": d["n_benign"],
                    "GridSybil": d["n_gridsybil"],
                    "AUROC": d["auroc"],
                    "PR-AUC": d["pr_auc"],
                    "recall p95": d["p95"]["recall"],
                    "benign FPR p95": d["p95"]["benign_fpr"],
                }
            )
pd.DataFrame(rows).set_index(["subset", "score", "group"]).round(4)

Benign false-positive rate per source folder at the val p95 / p99 window thresholds (all test windows). The same
benign traffic recurs in every family (F1); GridSybil_0709 is a separate re-simulation (F14).

In [ ]:
r = reports["all test windows"]["benign_fpr_by_folder"]
fpr = pd.DataFrame(
    {(name, q): {folder: d[q] for folder, d in r[name].items()} for name in SCORES for q in ("p95", "p99")}
)
fpr.insert(0, "benign windows", pd.Series({folder: d["n_benign"] for folder, d in r["reconstruction"].items()}))
fpr.round(4)

Score histograms per window (test, by class, read after scoring) with the val p95 threshold, and the median reconstruction error by window length.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, name in zip(axes, SCORES):
    s = np.log10(np.maximum(saved[name], 1e-6))
    bins = np.linspace(np.quantile(s, 0.001), np.quantile(s, 0.999), 80)
    for v, lab, c in ((0, "benign", "tab:blue"), (1, "GridSybil", "tab:red")):
        ax.hist(s[labels.y == v], bins=bins, alpha=0.55, density=True, label=lab, color=c)
    ax.axvline(np.log10(summary["thresholds"]["window"][name]["p95"]), color="k", ls="--", lw=1, label="val p95")
    ax.set_xlabel(f"log10 {name} score")
    ax.set_title(f"{name}: test windows", fontsize=10)
    ax.legend(fontsize=8)
df = pd.DataFrame({"n": saved["n_messages"], "e": saved["reconstruction"], "y": labels.y})
for v, lab, c in ((0, "benign", "tab:blue"), (1, "GridSybil", "tab:red")):
    g = df[df.y == v].groupby("n").e.median()
    axes[2].plot(g.index, g.values, marker=".", label=lab, color=c)
axes[2].set_yscale("log")
axes[2].set_xlabel("real messages per window")
axes[2].set_ylabel("median reconstruction error")
axes[2].set_title("reconstruction error by window length", fontsize=10)
axes[2].legend(fontsize=8)
fig.tight_layout()
plt.show()

Mean per-feature reconstruction error by class on the test windows, and the AUROC of each feature's error alone.

In [ ]:
ef = test_out["error_feature"]
feat = pd.DataFrame(
    {
        "mean error benign": ef[labels.y == 0].mean(0),
        "mean error GridSybil": ef[labels.y == 1].mean(0),
        "AUROC of the feature's error": [roc_auc_score(labels.y, ef[:, j]) for j in range(ef.shape[1])],
    },
    index=FEATURES,
)
feat.round(4)

## 6. Context

Numbers of the two T = 10 siblings (`task/autoencoder_st_t10`, `task/autoencoder_benign_t10`), read by
`evaluate.py` from the result cells of their executed notebooks (shown only when those notebooks have been
executed). The supervised study uses the same data, split and test windows; **the one-class study uses a different
split (`all/T10_benign`, 80 / 10 / 10) and test set**. One seed each: these numbers frame the result, they do not
rank the models.

In [ ]:
ctx = stored["context"]
a = reports["all test windows"]
rows = {
    "this study: unlabelled mix, reconstruction": {
        "data / split": "all/T10_benign_grid 85 / 7.5 / 7.5",
        "training signal": "none (benign + GridSybil pooled)",
        "window AUROC": a["window"]["reconstruction"]["auroc"],
    },
    "this study: unlabelled mix, Mahalanobis": {
        "data / split": "all/T10_benign_grid 85 / 7.5 / 7.5",
        "training signal": "none (benign + GridSybil pooled)",
        "window AUROC": a["window"]["mahalanobis"]["auroc"],
    },
}
if "supervised_st_t10" in ctx:
    rows["task/autoencoder_st_t10 (context)"] = {
        "data / split": "same data, split and test windows",
        "training signal": ctx["supervised_st_t10"]["training_signal"],
        "window AUROC": ctx["supervised_st_t10"]["window_auroc"],
    }
if "one_class_benign_t10" in ctx:
    rows["task/autoencoder_benign_t10, reconstruction (context)"] = {
        "data / split": "all/T10_benign 80 / 10 / 10 (different split and test set)",
        "training signal": ctx["one_class_benign_t10"]["training_signal"],
        "window AUROC": ctx["one_class_benign_t10"]["window_auroc_reconstruction"],
    }
pd.DataFrame(rows).T

The numbers the analysis below refers to.

In [ ]:
def brief(r):
    out = {
        "windows": r["windows"],
        "gridsybil_share": round(r["gridsybil_share"], 4),
        "length_only_auroc": round(r["length_only"]["auroc"], 4),
        "length_only_auroc_best_direction": round(r["length_only"]["auroc_best_direction"], 4),
    }
    for name in SCORES:
        w, t = r["window"][name], r["track"][name]
        out[name] = {
            "auroc": round(w["auroc"], 4),
            "pr_auc": round(w["pr_auc"], 4),
            "recall_p95": round(w["p95"]["recall"], 4),
            "benign_fpr_p95": round(w["p95"]["benign_fpr"], 4),
            "recall_p99": round(w["p99"]["recall"], 4),
            "benign_fpr_p99": round(w["p99"]["benign_fpr"], 4),
            "auroc_rows_1_3": round(w["rows_1_3"]["auroc"], 4),
            "auroc_rows_ge_4": round(w["rows_ge_4"]["auroc"], 4),
            "auroc_0709": round(w["by_group"]["0709"]["auroc"], 4),
            "auroc_1416": round(w["by_group"]["1416"]["auroc"], 4),
            "track_max_auroc": round(t["max"]["auroc"], 4),
            "track_mean_auroc": round(t["mean"]["auroc"], 4),
        }
    return out


results = {
    "run": RUN.name,
    "params": summary["params"],
    "best_epoch": summary["best_epoch"],
    "best_val_loss": round(summary["best_val_loss"], 5),
    "epochs_run": summary["epochs_run"],
    "stop_reason": summary["stop_reason"],
    "lr_changes": len(summary["lr_changes"]),
    "train_minutes": round(summary["train_seconds"] / 60, 1),
    "train_gridsybil_share": round(stored["class_counts"]["train"]["gridsybil_share"], 4),
    "val_gridsybil_share": round(stored["class_counts"]["val"]["gridsybil_share"], 4),
    "test_all": brief(reports["all test windows"]),
    "test_gridsybil_folders": brief(reports["GridSybil folders only"]),
}
results

## 7. Analysis

**Not written yet.** This notebook was copied from the T = 32 notebook before any T = 10 run existed; the T = 32
analysis text was removed because its numbers do not describe this run (RULE 1). Write the analysis from the
outputs above after `npm run ae10:train` and `npm run ae10:eval` has finished (one seed per run folder, no confidence interval; caveats F1, F11, F13,
F14 as in the T = 32 notebook).